[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Favioleiva/diarizator/blob/main/colab/Diarizator_Community1_Colab.ipynb)

# Diarizator Community-1 — upload-first Colab

**Standard workflow:** add `HF_TOKEN` to Colab Secrets, select **Run all**, upload one supported file, and download the sanitized ZIP. Google Drive is not required. Select a GPU runtime first.

Community-1 is gated: accept the conditions on the [official model page](https://huggingface.co/pyannote/speaker-diarization-community-1), create a read token, add it as the `HF_TOKEN` secret, and enable notebook access. Never paste a token into a cell.


### Step 1 — Install and validate the dependency profile

Run the code cell below by clicking the **▶** button on its left.

On the first execution, Diarizator may install or update the validated Python dependency profile. If that happens, the cell will stop intentionally and display a message asking you to restart the Colab session.

When you see that message:

1. Select **Runtime → Restart session**.
2. After the restart, run **this same dependency cell again**.
3. Continue only when the cell finishes with:

`DEPENDENCY_PROFILE_OK`

The first stop is expected and does **not** indicate an installation failure.


In [ ]:

# ============================================================
# DIARIZATOR COMMUNITY COLAB
# Unified Dependency Bootstrap v3
# Python + Hugging Face + CUDA 12 + GPU Preflight
# Safe two-pass restart guard
# ============================================================

import importlib.metadata as md
import json
import os
import shutil
import site
import subprocess
import sys
import urllib.request
from pathlib import Path

# ------------------------------------------------------------
# 1. Repository configuration
# ------------------------------------------------------------

OWNER = "Favioleiva"
REPOSITORY = "diarizator"
REF = "main"

REPO_URL = f"https://github.com/{OWNER}/{REPOSITORY}.git"
RAW = (
    f"https://raw.githubusercontent.com/"
    f"{OWNER}/{REPOSITORY}/{REF}/colab"
)

REQUIREMENTS = Path("/content/requirements-colab.txt")
CONSTRAINTS = Path("/content/constraints-colab.txt")
LOCAL_CONSTRAINTS = Path(
    "/content/constraints-colab-resolved.txt"
)

# ------------------------------------------------------------
# 2. Validated dependency profile
# ------------------------------------------------------------

EXPECTED = {
    "numpy": "2.2.2",
    "scipy": "1.16.3",
    "pandas": "2.2.3",
    "jedi": "0.19.2",
    "faster-whisper": "1.2.1",
    "ctranslate2": "4.8.1",
    "pyannote.audio": "4.0.7",
    "pyannote.metrics": "4.1",
    "opentelemetry-api": "1.42.1",
    "opentelemetry-sdk": "1.42.1",
    "tokenizers": "0.23.1",
    "fsspec": "2025.12.0",
    "huggingface-hub": "1.31.0",
    "transformers": "5.18.0",
}

CUDA_EXPECTED = {
    "nvidia-cublas-cu12": "12.9.2.10",
    "nvidia-cudnn-cu12": "9.27.0.42",
}

# Historical conflicts observed in the Colab image.
# Only these exact dependency-conflict patterns are tolerated.
KNOWN_CONFLICTS = (
    "google-colab 1.0.0 has requirement pandas==2.2.2",
    "numba 0.60.0 has requirement numpy<2.1,>=1.22",
)


# ------------------------------------------------------------
# 3. Utilities
# ------------------------------------------------------------

def version_of(package):
    try:
        return md.version(package)
    except md.PackageNotFoundError:
        return None


def pip_install(*args):
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", *args]
    )


def download_manifests():
    for filename, target in (
        ("requirements-colab.txt", REQUIREMENTS),
        ("constraints-colab.txt", CONSTRAINTS),
    ):
        urllib.request.urlretrieve(
            f"{RAW}/{filename}",
            target,
        )


def build_local_constraints():
    """
    Keep the repository constraints intact on GitHub.
    Override conflicting versions in a local copy.
    """

    overrides = {
        "huggingface-hub": "1.31.0",
        "tokenizers": "0.23.1",
        "fsspec": "2025.12.0",
        "transformers": "5.18.0",
        **CUDA_EXPECTED,
    }

    original = CONSTRAINTS.read_text().splitlines()
    resolved = []

    for line in original:
        stripped = line.strip()

        if (
            not stripped
            or stripped.startswith("#")
            or stripped.startswith("-")
        ):
            resolved.append(line)
            continue

        name = stripped.split("==")[0].strip().lower()
        name = name.replace("_", "-")

        if name in overrides:
            continue

        resolved.append(line)

    for package, version in overrides.items():
        resolved.append(f"{package}=={version}")

    LOCAL_CONSTRAINTS.write_text(
        "\n".join(resolved) + "\n"
    )

    print("LOCAL_CONSTRAINTS_OK")


def check_versions():
    all_expected = {**EXPECTED, **CUDA_EXPECTED}

    mismatches = {
        package: {
            "installed": version_of(package),
            "expected": expected,
        }
        for package, expected in all_expected.items()
        if version_of(package) != expected
    }

    return mismatches


def import_probe():
    code = """
import numpy
import scipy
import pandas
import tokenizers
import fsspec
import huggingface_hub
import transformers
import ctranslate2
import diarizator
print("BINARY_IMPORTS_OK")
"""

    result = subprocess.run(
        [sys.executable, "-c", code],
        capture_output=True,
        text=True,
    )

    return result.returncode == 0, result.stderr


# ------------------------------------------------------------
# 4. Dependency consistency check
# ------------------------------------------------------------

def verify_pip():
    result = subprocess.run(
        [sys.executable, "-m", "pip", "check"],
        capture_output=True,
        text=True,
    )

    output = result.stdout + "\n" + result.stderr

    unexpected = []

    for line in output.splitlines():
        line = line.strip()

        if not line or line == "No broken requirements found.":
            continue

        if not any(
            known in line for known in KNOWN_CONFLICTS
        ):
            unexpected.append(line)

    if unexpected:
        raise RuntimeError(
            "Unexpected dependency conflicts:\n"
            + "\n".join(unexpected)
        )

    if result.returncode != 0 and not output.strip():
        raise RuntimeError(
            "pip check failed without diagnostic output"
        )

    if result.returncode != 0:
        print(
            "WARNING: Allowlisted Colab conflicts remain."
        )

    print("PYTHON_DEPENDENCIES_OK")


# ------------------------------------------------------------
# 5. CUDA GPU preflight
# ------------------------------------------------------------

def cuda_library_paths():
    paths = []

    for root in site.getsitepackages():
        for name in ("cublas", "cudnn"):
            directory = (
                Path(root) / "nvidia" / name / "lib"
            )

            if directory.is_dir():
                paths.append(str(directory))

    return list(dict.fromkeys(paths))


def verify_cuda():
    print("\n=== CUDA PREFLIGHT ===")

    if not shutil.which("nvidia-smi"):
        raise RuntimeError(
            "NVIDIA GPU not detected. "
            "Select Runtime > Change runtime type > GPU."
        )

    gpu = subprocess.run(
        ["nvidia-smi", "-L"],
        capture_output=True,
        text=True,
    )

    if gpu.returncode != 0 or not gpu.stdout.strip():
        raise RuntimeError(
            "NVIDIA driver/GPU unavailable:\n"
            + gpu.stderr[-2000:]
        )

    print(gpu.stdout.strip())

    paths = cuda_library_paths()

    if len(paths) < 2:
        raise RuntimeError(
            "Missing cuBLAS or cuDNN library directories: "
            + json.dumps(paths)
        )

    env = os.environ.copy()

    env["LD_LIBRARY_PATH"] = os.pathsep.join(
        paths + [env.get("LD_LIBRARY_PATH", "")]
    )

    code = """
import ctypes
import ctranslate2

ctypes.CDLL("libcublas.so.12")
ctypes.CDLL("libcudnn.so.9")

count = ctranslate2.get_cuda_device_count()

assert count > 0, "No CUDA GPU detected by CTranslate2"

print("CUDA_LIBRARIES_OK")
print("CTRANSLATE2_GPU_COUNT:", count)
"""

    result = subprocess.run(
        [sys.executable, "-c", code],
        env=env,
        capture_output=True,
        text=True,
    )

    if result.returncode != 0:
        raise RuntimeError(
            "CUDA preflight failed:\n"
            + result.stderr[-3000:]
        )

    print(result.stdout.strip())

    return paths


# ------------------------------------------------------------
# 6. Main bootstrap
# ------------------------------------------------------------

def setup_dependencies():
    print("=== DIARIZATOR UNIFIED BOOTSTRAP ===")

    download_manifests()
    build_local_constraints()

    mismatches = check_versions()
    imports_ok, import_error = import_probe()

    # Detect whether the active environment needs repair.
    needs_install = bool(mismatches) or not imports_ok

    if needs_install:
        print("\nDependency repair required.")

        if mismatches:
            print(json.dumps(mismatches, indent=2))

        # Preserve the NumPy/SciPy binary profile.
        pip_install(
            "--upgrade",
            "--force-reinstall",
            "numpy==2.2.2",
            "scipy==1.16.3",
        )

        # Install repository requirements using corrected
        # local constraints.
        pip_install(
            "-r", str(REQUIREMENTS),
            "-c", str(LOCAL_CONSTRAINTS),
        )

        # Install all explicit overrides consistently.
        overrides = {
            "tokenizers": "0.23.1",
            "fsspec": "2025.12.0",
            "huggingface-hub": "1.31.0",
            "transformers": "5.18.0",
            **CUDA_EXPECTED,
        }

        pip_install(
            "--no-deps",
            *[
                f"{name}=={version}"
                for name, version in overrides.items()
            ],
        )

        # Install Diarizator source.
        pip_install(
            "--no-deps",
            f"diarizator @ git+{REPO_URL}@{REF}",
        )

        # Validate package versions and binary imports.
        remaining = check_versions()
        imports_ok, import_error = import_probe()

        if remaining:
            raise RuntimeError(
                "Dependency versions still differ:\n"
                + json.dumps(remaining, indent=2)
            )

        if not imports_ok:
            raise RuntimeError(
                "Binary import verification failed:\n"
                + import_error[-3000:]
            )

        verify_pip()

        print("\n✅ INSTALLATION_COMPLETE")
        print("✅ BINARY_IMPORTS_OK")
        print("\nRESTART REQUIRED")
        print("1. Runtime > Restart session")
        print("2. Run THIS SAME CELL again")
        print("3. Continue when DEPENDENCY_PROFILE_OK appears")

        return False

    # The package profile already matches.
    verify_pip()

    # Validate CUDA in a fresh Python process.
    paths = verify_cuda()

    global CUDA_LIBRARY_PATHS
    CUDA_LIBRARY_PATHS = paths

    print("\n=== FINAL PACKAGE PROFILE ===")

    for package, expected in {
        **EXPECTED,
        **CUDA_EXPECTED,
    }.items():
        print(f"{package}: {version_of(package)}")

    print("\n✅ BINARY_IMPORTS_OK")
    print("✅ PYTHON_DEPENDENCIES_OK")
    print("✅ CUDA_RUNTIME_OK")
    print("✅ DEPENDENCY_PROFILE_OK")
    print("✅ READY_FOR_DIARIZATION")

    return True


DEPENDENCIES_READY = setup_dependencies()


## Optional configuration

The default is **Use bundled demonstration audio** with automatic speaker-count estimation. Change it to **"Upload my own audio"** for your own audio. Advanced users may choose exact or bounded counts. The bundled demonstration may use its known exact two-character structure; arbitrary uploads never silently use that setting.


In [ ]:
INPUT_CHOICE = "Upload my own audio"  # or: Upload my own audio
SPEAKER_COUNT_MODE = "estimated"  # estimated, exact, or bounded
NUM_SPEAKERS = None
MIN_SPEAKERS = None
MAX_SPEAKERS = None
DEMO_SPEAKER_COUNT_MODE = "exact"  # exact or estimated
if SPEAKER_COUNT_MODE not in {"estimated", "exact", "bounded"}: raise ValueError("Invalid speaker-count mode")
if SPEAKER_COUNT_MODE == "exact" and (not isinstance(NUM_SPEAKERS, int) or NUM_SPEAKERS < 1): raise ValueError("Exact mode requires NUM_SPEAKERS >= 1")
if SPEAKER_COUNT_MODE == "bounded" and (not isinstance(MIN_SPEAKERS, int) or not isinstance(MAX_SPEAKERS, int) or MIN_SPEAKERS < 1 or MAX_SPEAKERS <= MIN_SPEAKERS): raise ValueError("Bounded mode requires 1 <= MIN_SPEAKERS < MAX_SPEAKERS")


In [ ]:
# Secure authentication and Community-1 access preflight happen before ASR.
from google.colab import userdata
from diarizator.access import preflight_model_access
HF_TOKEN = userdata.get("HF_TOKEN")
access = preflight_model_access(token=HF_TOKEN)
print("Community-1 access:", access["status"])
if access["status"] not in {"MODEL_CACHED", "MODEL_ACCESS_CONFIRMED"}:
    raise RuntimeError("Community-1 preflight stopped safely: " + access["status"])


## Select one input

Supported formats include MP3, M4A/AAC, WAV, FLAC, OGG/OPUS, and audio tracks from common MP4, MOV, MKV, and WebM files. For faster browser uploads, a compressed format such as MP3 is recommended. M4A/AAC is also compact. WAV files are usually much larger. This is about upload size—not accuracy.


In [ ]:
from pathlib import Path
import shutil, subprocess, sys, tempfile
from google.colab import files
workspace = Path(tempfile.mkdtemp(prefix="diarizator-"))
if INPUT_CHOICE == "Upload my own audio":
    uploaded = files.upload()
    if len(uploaded) != 1: raise ValueError("Upload exactly one supported audio or video file.")
    filename, payload = next(iter(uploaded.items()))
    audio_path = workspace / Path(filename).name
    audio_path.write_bytes(payload)
    mode, number, lower, upper = SPEAKER_COUNT_MODE, NUM_SPEAKERS, MIN_SPEAKERS, MAX_SPEAKERS
elif INPUT_CHOICE == "Use bundled demonstration audio":
    source = "https://raw.githubusercontent.com/Favioleiva/diarizator/main/examples/controlled_demo/The_Measure_of_a_Good_Life.mp3"
    audio_path = workspace / "The_Measure_of_a_Good_Life.mp3"
    urllib.request.urlretrieve(source, audio_path)
    mode = DEMO_SPEAKER_COUNT_MODE
    number, lower, upper = (2 if mode == "exact" else None), None, None
else: raise ValueError("Unknown input choice")
from diarizator.media import inspect_media
print(json.dumps(inspect_media(audio_path), indent=2))


In [ ]:

# ============================================================
# DIARIZATOR COMMUNITY COLAB
# CUDA-safe diarization + ZIP export
# ============================================================

import os
import sys
import json
import site
import subprocess
import tempfile
from pathlib import Path

from google.colab import files
from diarizator.bundle import create_bundle

# ------------------------------------------------------------
# 1. Validate inputs
# ------------------------------------------------------------

if not DEPENDENCIES_READY:
    raise RuntimeError(
        "Dependencies are not ready. "
        "Complete the bootstrap and restart procedure."
    )

audio_path = Path(audio_path).resolve()
workspace = Path(workspace).resolve()

if not audio_path.is_file():
    raise FileNotFoundError(audio_path)

if not HF_TOKEN:
    raise RuntimeError("Hugging Face token is missing.")

# ------------------------------------------------------------
# 2. Configure CUDA library search paths
# ------------------------------------------------------------

cuda_paths = []

for root in site.getsitepackages():
    for name in ("cublas", "cudnn"):
        directory = Path(root) / "nvidia" / name / "lib"

        if directory.is_dir():
            cuda_paths.append(str(directory))

cuda_paths = list(dict.fromkeys(cuda_paths))

if len(cuda_paths) < 2:
    raise RuntimeError(
        "CUDA library directories not found. "
        "Run the dependency bootstrap first."
    )

env = os.environ.copy()

env["LD_LIBRARY_PATH"] = os.pathsep.join(
    cuda_paths + [env.get("LD_LIBRARY_PATH", "")]
)

# Token is passed through the environment,
# not through command-line arguments.
env["DIARIZATOR_HF_TOKEN"] = HF_TOKEN

# ------------------------------------------------------------
# 3. Prepare isolated worker
# ------------------------------------------------------------

settings = {
    "audio_path": str(audio_path),
    "output_dir": str(workspace / "results"),
    "speaker_count_mode": mode,
    "num_speakers": number,
    "min_speakers": lower,
    "max_speakers": upper,
}

worker_script = r"""
import os
import sys
import json
from pathlib import Path

from diarizator import diarize

settings = json.loads(sys.stdin.read())

result_dir = diarize(
    Path(settings["audio_path"]),
    Path(settings["output_dir"]),
    token=os.environ["DIARIZATOR_HF_TOKEN"],
    speaker_count_mode=settings["speaker_count_mode"],
    num_speakers=settings["num_speakers"],
    min_speakers=settings["min_speakers"],
    max_speakers=settings["max_speakers"],
)

if not Path(result_dir).is_dir():
    raise RuntimeError("Diarizator returned no result directory")

print("DIARIZATION_COMPLETE", flush=True)
"""

# ------------------------------------------------------------
# 4. Run diarization with CUDA available at process startup
# ------------------------------------------------------------

print("=== DIARIZATOR CUDA WORKER ===")
print("Audio:", audio_path.name)
print("CUDA libraries:", cuda_paths)
print("Starting transcription and diarization...",
      flush=True)

process = subprocess.run(
    [sys.executable, "-u", "-c", worker_script],
    input=json.dumps(settings),
    text=True,
    env=env,
    check=False,
)

if process.returncode != 0:
    raise RuntimeError(
        f"Diarizator worker failed "
        f"(exit code {process.returncode}). "
        "See the worker traceback above."
    )

# ------------------------------------------------------------
# 5. Locate completed results
# ------------------------------------------------------------

results_root = workspace / "results"

run_files = list(results_root.rglob("run.json"))

if not run_files:
    raise RuntimeError(
        "No run.json found after diarization."
    )

# Choose the most recently written run.
run_file = max(
    run_files,
    key=lambda p: p.stat().st_mtime
)

result_dir = run_file.parent

run = json.loads(
    (result_dir / "run.json").read_text()
)

environment = json.loads(
    (result_dir / "environment.json").read_text()
)

diarization = json.loads(
    (result_dir / "diarization.json").read_text()
)

if str(run.get("status", "")).upper() != "COMPLETED":
    raise RuntimeError(
        "Diarizator did not complete successfully: "
        + str(run.get("status"))
    )

# ------------------------------------------------------------
# 6. Create result bundle
# ------------------------------------------------------------

bundle_path = workspace / "diarizator-results.zip"

bundle = create_bundle(
    [result_dir],
    bundle_path
)

# ------------------------------------------------------------
# 7. Print execution report
# ------------------------------------------------------------

report = {
    "zip_filename": bundle_path.name,
    "zip_path": str(bundle_path),
    "zip_size_bytes": bundle_path.stat().st_size,
    "zip_sha256": bundle["sha256"],
    "detected_gpu": environment.get("device"),
    "backend": (
        run.get("config", {})
        .get("diarization", {})
        .get("backend")
    ),
    "speaker_count_mode": run.get(
        "speaker_count_mode"
    ),
    "detected_speaker_count": diarization.get(
        "estimated_speakers"
    ),
    "run_status": run.get("status"),
}

print("\n=== DIARIZATOR EXECUTION REPORT ===")
print(json.dumps(report, indent=2))

# ------------------------------------------------------------
# 8. Download ZIP
# ------------------------------------------------------------

print("\nDownloading results...")
files.download(str(bundle_path))


In [ ]:
from diarizator.bundle import create_bundle
from google.colab import files
import json

# Validate completed execution
assert str(run.get("status", "")).upper() == "COMPLETED", (
    f"Unexpected run status: {run.get('status')}"
)

# Create bundle
bundle_path = workspace / "diarizator-results.zip"
bundle = create_bundle([result_dir], bundle_path)

# Execution report
report = {
    "zip_filename": bundle_path.name,
    "zip_path": str(bundle_path),
    "zip_size_bytes": bundle_path.stat().st_size,
    "zip_sha256": bundle["sha256"],
    "detected_gpu": environment.get("device"),
    "backend": run.get("config", {})
                  .get("diarization", {})
                  .get("backend"),
    "speaker_count_mode": run.get("speaker_count_mode"),
    "detected_speaker_count": diarization.get(
        "estimated_speakers"
    ),
    "run_status": run.get("status"),
}

print(json.dumps(report, indent=2))

# Download results
files.download(str(bundle_path))